# Baseline Tasks

> This section provides an overview of the baseline tasks for benchmarking nanobot capabilities.

In [ ]:
#| default_exp baseline

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from datasets import load_dataset
from pathlib import Path
from pydantic import BaseModel, Field
from typing import List, Dict, Literal, Any, Optional
import urllib.request
import json

## Baseline Task Template

For our experiment setup, we will use a fixed template for all baseline tasks. The template is adopted from the minimal fields in project plan, with three major deviations:  

- For reference answers, we will save all information in a subschema called `ReferenceContext` instead of a single string of final answer. This allows us to store more information and adapt to different answer formats from the sources (e.g., tool calls from BCFL; final answer and annotator notes from GAIA).  
- For tools, we save the task-specific tools in a subschema called `ToolConfig` instead of a list of available tools. This tool configuration either points to "default", which is the default tool set from `nanobot`, or a list of tool schema for that specific task. This is to adapt to BCFL's setting where each task has its own tool set.  
- For evaluation, we add another matching mode into `GoalCondition` called `rubric_eval`, which is for LLM evaluation based on a rubric. This is to adapt to GAIA's dataset where evaluation from exact match may not be sufficient (e.g., dependent on search engines)


The template declares the necessary metadata for tasks, including the fields as following:

For default toolbox, we will use the nanobot toolbox, which includes the following tasks:

- `cron`: Schedule reminders and recurring tasks.  
- `edit_file`: Edit a file by replacing text.  
- `exec`: Execute shell commands.  
- `glob`: Find files matching a glob pattern.  
- `grep`: Search file contents with regex or plain text.  
- `list_dir`: List directory contents (recursively).  
- `message`: Send messages (including file attachments).  
- `read_file`: Read file contents (with pagination).  
- `spawn`: Spawn a subagent for background tasks.  
- `web_fetch`: Fetch and extract content from URLs.  
- `web_search`: Search the web for titles, URLs, and snippets.  
- `write_file`: Write content to a file.  

In [ ]:
#| export
# --- Literal Definitions ---
SourceEnum = Literal["GAIA", "BFCL", "tau-bench"]               # Dataset or origin of the task
ToolCountClassEnum = Literal["single_tool", "multi_tool"]       # Number of tools required for task completion
CompositionDepthEnum = Literal["1", "2_3", "4_plus"]            # Depth of reasoning/composition required
PlanningHorizonEnum = Literal["short", "medium", "long"]        # Temporal aspect of the task (e.g., number of steps, time to completion)
ScaffoldTypeEnum = Literal["memory", "prompt_scaffold", \
                           "multi_agent", "combined"]           # Types of scaffolding available
ToolSourceEnum = Literal["default", "custom"]                   # Source of the tools provided for the task

# Methods for evaluating task completion
GoalConditionTypeEnum = Literal[
    "text_match", 
    "file_creation", 
    "json_state_match", 
    "exit_code", 
    "answer_set_match",
    "llm_judge" 
]

# Key Performance Indicators (KPIs) this task is designed to evaluate
KPIEnum = Literal[
    "Reasoning-First Success Rate",
    "Soft-Scaffolding Gain",
    "Fallback Justification Rate",
    "Task Completion Rate",
    "Tool Selection Accuracy",
    "Decomposition Stability",
    "Retry Count per Task",
    "Retry Improvement",
    "Context Degradation",
    "Hallucination Rate",
    "Autonomous Duration",
    "Plan Length vs Failure"
]

# --- Nested Pydantic Models ---
class ReferenceContext(BaseModel):
    """Stores ground-truth data, expected paths, and annotator warnings for the evaluator."""
    expected_tool_calls: Optional[List[Dict[str, Any]]] = Field(
        default=None, description="Expected JSON payloads for tools (e.g., from BFCL)"
    )
    annotator_notes: Optional[str] = Field(
        default=None, description="Hints, trajectories, or temporal warnings from datasets like GAIA"
    )
    expected_final_answer: Optional[str] = Field(
        default=None, description="The final conclusion or string the agent should reach"
    )

class ToolConfig(BaseModel):
    """Encapsulates all logic for how the agent receives its tools."""
    source: ToolSourceEnum = Field(
        default="default",
        description="Flag to tell the evaluator which toolset to load."
    )
    custom_tools: Optional[List[Dict[str, Any]]] = Field(
        default=None, 
        description="Only populated if source is 'custom' (e.g., BFCL mock schemas)."
    )

class GoalCondition(BaseModel):
    type: GoalConditionTypeEnum
    target: Any  # Can be a string, dict, int, or point to "reference_context"
    # Match modes for text-based conditions or LLM evaluation criteria
    match_mode: Literal["exact", "bounded", "includes", "rubric_eval"] = Field(
        default="exact", description="How to evaluate the goal condition for text-based tasks"
    )
    eval_rubric: Optional[str] = Field(
        default=None, description="Explicit instructions for an LLM Judge"
    )

class InterventionPolicy(BaseModel):
    reasoning_first: bool = True
    soft_scaffold_allowed: List[ScaffoldTypeEnum] = Field(default_factory=list)
    hard_fallback_allowed: bool = False

# --- Main Baseline Task Schema ---
class BaselineTaskSchema(BaseModel):
    """
    Schema for tasks defined in baseline.json
    """
    id: str
    source: SourceEnum
    source_split: str
    task_family: str
    difficulty: str
    prompt: str

    primary_kpi: List[KPIEnum]

    # Required task-shape metadata
    tool_count_class: ToolCountClassEnum
    composition_depth: CompositionDepthEnum
    planning_horizon: PlanningHorizonEnum
    cross_api_mashup: bool

    # Upgraded to accept full JSON schemas (OpenAI style) instead of just string labels
    tool_config: ToolConfig = Field(default_factory=ToolConfig)

    # New Vault for Ground Truth and Context
    reference_context: Optional[ReferenceContext] = None

    goal_condition: GoalCondition
    
    judge_mode: Literal["automatic", "manual", "llm_evaluator"] = "automatic"
    risk_tags: List[str]
    intervention_policy: InterventionPolicy

In [ ]:
#| echo: false
print("Baseline Task Schema defined successfully!")
print(json.dumps(BaselineTaskSchema.model_json_schema(), indent=2))

Baseline Task Schema defined successfully!
{
  "$defs": {
    "GoalCondition": {
      "properties": {
        "type": {
          "enum": [
            "text_match",
            "file_creation",
            "json_state_match",
            "exit_code",
            "answer_set_match",
            "llm_judge"
          ],
          "title": "Type",
          "type": "string"
        },
        "target": {
          "title": "Target"
        },
        "match_mode": {
          "default": "exact",
          "description": "How to evaluate the goal condition for text-based tasks",
          "enum": [
            "exact",
            "bounded",
            "includes",
            "rubric_eval"
          ],
          "title": "Match Mode",
          "type": "string"
        },
        "eval_rubric": {
          "anyOf": [
            {
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "desc

Save and load tasks to/from JSON files:

In [ ]:
#| export
def save_baseline_tasks(baseline_tasks: List[BaselineTaskSchema], path: Path | str = "data/baseline_tasks.json"):
    path = Path(path)
    if not path.parent.exists():
        path.parent.mkdir(parents=True)
    with open(path, "w") as f:
        json.dump([task.model_dump() for task in baseline_tasks], f, indent=2)

def load_baseline_tasks(path: Path | str = "data/baseline_tasks.json") -> List[BaselineTaskSchema]:
    path = Path(path)
    with open(path, "r") as f:
        data = json.load(f)
    return [BaselineTaskSchema(**item) for item in data]

## GAIA Task Selection

One of our sources of tasks is [GAIA](https://gaia-bench.github.io/), which provides a large collection of real-world multi-step tasks. We will filter the GAIA tasks based on the following criteria:

- **Level**: We will focus on tasks from `level_1` and `level_2`, which are designed to require multi-step reasoning and tool use.  
- **Tools**: We will select tasks that match the default tool set available from `nanobot` 

In [ ]:
#| export
def fetch_gaia():
    print("Fetching GAIA data...")
    # Load the 2023 validation split
    dataset = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")
    return dataset

In [ ]:
#| eval: False
# Test it out
gaia_tasks = fetch_gaia()
print(f"Number of tasks: {len(gaia_tasks)}")
print("Sample task:")
print(json.dumps(gaia_tasks[0], indent=2))

Fetching GAIA data...
Number of tasks: 165
Sample task:
{
  "task_id": "c61d22de-5f6c-4958-a7f6-5e9707bd3466",
  "Question": "A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure with three axes, where each axis has a label word at both ends. Which of these words is used to describe a type of society in a Physics and Society article submitted to arXiv.org on August 11, 2016?",
  "Level": "2",
  "Final answer": "egalitarian",
  "file_name": "",
  "file_path": "",
  "Annotator Metadata": {
    "Steps": "1. Go to arxiv.org and navigate to the Advanced Search page.\n2. Enter \"AI regulation\" in the search box and select \"All fields\" from the dropdown.\n3. Enter 2022-06-01 and 2022-07-01 into the date inputs, select \"Submission date (original)\", and submit the search.\n4. Go through the search results to find the article that has a figure with three axes and labels on each end of the axes, titled \"Fairness in Agreement With European Values

We check for tools which are not supported by `nanobot` and filter out tasks that require those tools. The unsupported tools include:

In [ ]:
#| eval: false
import re

tools = set()
for task in gaia_tasks:
    if "Annotator Metadata" in task and "Tools" in task["Annotator Metadata"]:
        task_tools = task["Annotator Metadata"]["Tools"].lower().split("\n")
        tools.update([re.sub(r'^\d+\.\s*', '', item) for item in task_tools])
print("Tools used in tasks:")
for tool in tools:
    print(f"- {tool}")

Tools used in tasks:
- (optional) web browser
- image recognition/ocr
- powerpoint viewer
- image recognition and processing tools
- search engine
- microsoft excel
- excel file access
- a speech-to-text tool
- access to the internet archive, web.archive.org
- text editor
- xlsx file access
- youtube
- pdf reader 
- none
- xml file access
- a search engine.
- a python ide
- image recognition software
- text processing/diff tool
- a speech-to-text audio processing tool
- word document access
- access to academic journal websites
- bablyonian cuniform -> arabic legend
- image recognition tools
- image recognition tools (to identify and parse a figure with three axes)
- access to wikipedia
- google search
- microsoft excel / google sheets
- natural language processor
- excel
- youtube player
- ocr
- csv file access
- a web browser.
- tool to extract text from images
- audio processing software
- video recognition tools
- counter
- computer algebra system
- calculator (or ability to count)

In [ ]:
#| eval: false
excluded_tools = [
    # Office & Binary Files
    "excel", "spreadsheet", "pdf", "powerpoint", "word document", 
    # Video & Audio
    "video", "youtube", "audio", "speech", 
    # Image & Vision
    "image", "vision", "color", "ocr", "gif", 
    # Highly specific visual/UI apps
    "rubik", "cuniform", "graph interaction", "maps"
]

In [ ]:
#| export
def filter_gaia_tasks(
    tasks: list[dict],  # List of GAIA tasks
    levels: list[int] = [1, 2, 3],  # Difficulty levels to include
    included_tools: list[str] = None,  # Tools that must be included
    excluded_tools: list[str] = None,  # Tools that must not be included
    maximum_steps: int = None,  # Maximum number of steps in the solution
) -> list[dict]:
    """
    Filter GAIA tasks based on specified criteria.
    
    Args:
        tasks: List of GAIA task dictionaries.
        levels: List of difficulty levels to include (default: [1, 2, 3]).
        included_tools: List of tools that must be included in the solution (default: None).
        excluded_tools: List of tools that must not be included in the solution (default: None).
        maximum_steps: Maximum number of steps allowed in the solution (default: None).
    
    Returns:
        A list of filtered GAIA task dictionaries.
    """
    filtered = []
    for task in tasks:
        # Check difficulty level
        if int(task['Level']) not in levels:
            continue
        
        tools = task['Annotator Metadata']['Tools'] \
            if 'Annotator Metadata' in task and 'Tools' in task['Annotator Metadata'] else []
        
        # Check included tools
        if included_tools is not None:
            if not any(tool in tools.lower() for tool in included_tools):
                continue
        
        # Check excluded tools
        if excluded_tools is not None:
            if any(tool in tools.lower() for tool in excluded_tools):
                continue
        
        # Check maximum steps
        if maximum_steps is not None:
            if 'Annotator Metadata' in task and 'Number of steps' in task['Annotator Metadata']:
                if int(task['Annotator Metadata']['Number of steps']) > maximum_steps:
                    continue
        
        filtered.append(task)
    
    return filtered

Filter GAIA tasks based on the above criteria and select a subset of tasks for our baseline evaluation. The selected tasks will be converted into our baseline task template format for consistency in evaluation.

In [ ]:
#| eval: False
filtered_tasks = filter_gaia_tasks(gaia_tasks, levels=[1, 2], excluded_tools=excluded_tools)
print(f"Number of filtered tasks: {len(filtered_tasks)}")
print("Sample filtered task:")
print(json.dumps(filtered_tasks[0], indent=2))

Number of filtered tasks: 80
Sample filtered task:
{
  "task_id": "17b5a6a3-bc87-42e8-b0fb-6ab0781ef2cc",
  "Question": "I\u2019m researching species that became invasive after people who kept them as pets released them. There\u2019s a certain species of fish that was popularized as a pet by being the main character of the movie Finding Nemo. According to the USGS, where was this fish found as a nonnative species, before the year 2020? I need the answer formatted as the five-digit zip codes of the places the species was found, separated by commas if there is more than one place.",
  "Level": "2",
  "Final answer": "34689",
  "file_name": "",
  "file_path": "",
  "Annotator Metadata": {
    "Steps": "1. Search the web for \u201cfinding nemo main character\u201d.\n2. Note the results, which state that the main character is a clownfish.\n3. Search the web for \u201cusgs nonnative species database\u201d.\n4. Click result for the Nonindigenous Aquatic Species site.\n5. Click \u201cMarine Fi

In [ ]:
#| export
def map_gaia_task(task: dict) -> BaselineTaskSchema:
    """Translates a raw GAIA task into the strict BaselineTaskSchema."""
    if "Annotator Metadata" not in task:
        print(f"Warning: Task {task.get('task_id', 'unknown')} is missing 'Annotator Metadata'. Defaulting to level 1 assumptions.")
        level = 1
    else:
        level = int(task["Annotator Metadata"].get("Level", 1))
    
    # 1. Level-based Heuristics for the Enums
    if level == 1:
        diff = "easy"
        depth = "1"
        horizon = "short"
        tool_class = "single_tool"
        mashup = False
    elif level == 2:
        diff = "medium"
        depth = "2_3"
        horizon = "medium"
        tool_class = "multi_tool"
        mashup = True
    else:  # Level 3
        diff = "hard"
        depth = "4_plus"
        horizon = "long"
        tool_class = "multi_tool"
        mashup = True

    # 2. Reference Context Extraction
    if "Annotator Metadata" in task:
        reference_context = ReferenceContext(
            expected_tool_calls=None,
            annotator_notes=task["Annotator Metadata"].get("Steps", None),
            expected_final_answer=task.get("Final answer", None)
        )
    else:
        reference_context = None

    # 3. Instantiate and Validate
    return BaselineTaskSchema(
        id=task.get("task_id", "unknown"),
        source="GAIA",
        source_split=f"level_{level}",
        
        # GAIA covers many domains, but for a default bucket:
        task_family="general_reasoning", 
        difficulty=diff,
        prompt=task.get("Question", ""),
        
        # Aligned with your KPI-to-benchmark mapping for GAIA
        primary_kpi=["Task Completion Rate", "Decomposition Stability"],
        
        tool_count_class=tool_class,
        composition_depth=depth,
        planning_horizon=horizon,
        cross_api_mashup=mashup,
        reference_context=reference_context,
        tool_config=ToolConfig(source="default"),  # Assuming we'll map GAIA tools to a default set
        
        goal_condition={
            "type": "eval_rubric" if level >= 2 else "text_match",
            "target": str(task.get("Final answer", "")),
            "match_mode": "exact"
        },
        
        judge_mode="automatic",
        risk_tags=["multi_step", "context_fragility"] if level >= 2 else ["single_step"],
        
        intervention_policy={
            "reasoning_first": True,
            "soft_scaffold_allowed": ["prompt_scaffold", "memory", "combined"],
            "hard_fallback_allowed": False
        }
    )

Map GAIA's suitable tasks into our baseline task template format and save them as JSON files:

In [ ]:
#| eval: false
baseline_tasks = [map_gaia_task(task) for task in filtered_tasks]
print(f"Number of baseline tasks: {len(baseline_tasks)}")

Number of baseline tasks: 80


In [ ]:
#| eval: false
save_baseline_tasks(baseline_tasks, "../data/gaia_subset.json")

## BCFL Task Selection

Another source of tasks is the [BCFL](https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard) dataset, which contains a variety of function calling tasks.

In [ ]:
#| export
def load_raw_data(url: str) -> list[dict]:
    """
    Fetches a specific BFCL category file directly from the Hugging Face raw dataset repo
    and loads it line-by-line as a list of JSON objects.
    """
    print(f"Fetching data from {url}...")
    
    try:
        # Hugging Face requires a User-Agent header to allow the download
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        response = urllib.request.urlopen(req)
        
        result = []
        # Iterate over the bytes returned by the HTTP response
        for line_bytes in response:
            line = line_bytes.decode('utf-8').strip()
            # Skip any accidental empty lines
            if line:  
                result.append(json.loads(line))
                
        print(f"Successfully loaded {len(result)} items from the dataset.")
        return result
        
    except Exception as e:
        print(f"Failed to fetch or load data: {e}")
        return []

def fetch_bfcl_data(category: str = "simple") -> list[dict]:
    """
    Fetches a specific BFCL category file directly from the Hugging Face raw dataset repo
    and loads it line-by-line as a list of JSON objects.
    """
    # Updated URL to point to the raw V3 files hosted on Hugging Face
    url = f"https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/main/BFCL_v3_{category}.json"
    answer_url = f"https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/main/possible_answer/BFCL_v3_{category}.json"

    print(f"Fetching BFCL '{category}' data from Hugging Face...")
    
    try:
        prompts = load_raw_data(url)
        answers = load_raw_data(answer_url)
        
        # Assuming both files have the same number of entries and are aligned
        if len(prompts) != len(answers):
            print(f"Warning: Mismatch in number of prompts ({len(prompts)}) and answers ({len(answers)}).")
        
        # Combine prompts and answers into a single list of dicts
        combined = []
        for i in range(min(len(prompts), len(answers))):
            current_id = prompts[i].get("id", f"item_{i}")
            if current_id != answers[i].get("id", f"item_{i}"):
                print(f"Warning: ID mismatch at index {i} (prompt ID: {current_id}, answer ID: {answers[i].get('id', 'unknown')}). Skipping this pair.")
                continue
            combined.append({
                "question": prompts[i]["question"],
                "function": prompts[i]["function"],
                "ground_truth": answers[i]["ground_truth"]
            })
        
        print(f"Successfully combined {len(combined)} prompt-answer pairs.")
        return combined
        
        
    except Exception as e:
        print(f"Failed to fetch or load BFCL data: {e}")
        return []

In [ ]:
#| eval: false
bfcl_simple_data = fetch_bfcl_data("simple")

if bfcl_simple_data:
    print("Sample Task:")
    print(json.dumps(bfcl_simple_data[0], indent=2))

Fetching BFCL 'simple' data from Hugging Face...
Fetching data from https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/main/BFCL_v3_simple.json...
Successfully loaded 400 items from the dataset.
Fetching data from https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/main/possible_answer/BFCL_v3_simple.json...
Successfully loaded 400 items from the dataset.
Successfully combined 400 prompt-answer pairs.
Sample Task:
{
  "question": [
    [
      {
        "role": "user",
        "content": "Find the area of a triangle with a base of 10 units and height of 5 units."
      }
    ]
  ],
  "function": [
    {
      "name": "calculate_triangle_area",
      "description": "Calculate the area of a triangle given its base and height.",
      "parameters": {
        "type": "dict",
        "properties": {
          "base": {
            "type": "integer",
            "description": "The base of the triangle."
          }

In [ ]:
#| export
def map_bfcl_task(task: dict, split: str, difficulty: str) -> BaselineTaskSchema:
    """Translates a raw BFCL task into the strict BaselineTaskSchema."""
    # 1. Basic Mappings
    difficulty = difficulty.lower()
    if difficulty not in ["easy", "medium", "hard"]:
        difficulty = "easy"  # Default to easy if unknown

    # 2. Reference Context Extraction
    reference_context = ReferenceContext(
        expected_tool_calls=task.get("ground_truth", None),
        annotator_notes=None,
        expected_final_answer=None
    )

    # 3. Tool Configuration Extraction
    # Assuming the "function" field contains the necessary tool information in a structured format
    tools = task.get("function", [])
    tool_config = ToolConfig(
        source="custom",
        custom_tools=tools if tools else None
    )

    # 4. Instantiate and Validate
    return BaselineTaskSchema(
        id=task.get("id", "unknown"),
        source="BFCL",
        source_split=split,
        task_family="code_generation",  # Assuming BFCL tasks are primarily code generation
        difficulty=difficulty,
        prompt=task["question"][0][0]["content"] if "question" in task and isinstance(task["question"], list) \
            and len(task["question"]) > 0 and isinstance(task["question"][0], list) and len(task["question"][0]) > 0 \
            and "content" in task["question"][0][0] else "",
        primary_kpi=["Tool Selection Accuracy", "Task Completion Rate"],
        tool_count_class="single_tool" if len(tools) <= 1 else "multi_tool",
        composition_depth="1" if difficulty == "easy" else ("2_3" if difficulty == "medium" else "4_plus"),
        planning_horizon="short" if difficulty == "easy" else ("medium" if difficulty == "medium" else "long"),
        cross_api_mashup=False,  # Assuming BFCL tasks do not require cross-API interactions
        reference_context=reference_context,
        tool_config=tool_config,
        goal_condition={
            "type": "json_state_match",
            "target": "reference_context.expected_tool_calls",
            "match_mode": "exact"
        },
        judge_mode="automatic",
        risk_tags=["code_generation", "tool_usage"],
        intervention_policy={
            "reasoning_first": True,
            "soft_scaffold_allowed": ["prompt_scaffold"],
            "hard_fallback_allowed": False
        }
    )

In [ ]:
#| eval: false
mapped_bfcl_tasks = [map_bfcl_task(task, split="simple", difficulty="medium") for task in bfcl_simple_data]
print(f"Number of mapped BFCL tasks: {len(mapped_bfcl_tasks)}")
print("Sample Mapped Task:")
print(mapped_bfcl_tasks[0].model_dump_json(indent=2))

Number of mapped BFCL tasks: 400
Sample Mapped Task:
{
  "id": "unknown",
  "source": "BFCL",
  "source_split": "simple",
  "task_family": "code_generation",
  "difficulty": "medium",
  "prompt": "Find the area of a triangle with a base of 10 units and height of 5 units.",
  "primary_kpi": [
    "Tool Selection Accuracy",
    "Task Completion Rate"
  ],
  "tool_count_class": "single_tool",
  "composition_depth": "2_3",
  "planning_horizon": "medium",
  "cross_api_mashup": false,
  "tool_config": {
    "source": "custom",
    "custom_tools": [
      {
        "name": "calculate_triangle_area",
        "description": "Calculate the area of a triangle given its base and height.",
        "parameters": {
          "type": "dict",
          "properties": {
            "base": {
              "type": "integer",
              "description": "The base of the triangle."
            },
            "height": {
              "type": "integer",
              "description": "The height of the triang

In [ ]:
#| eval: false
save_baseline_tasks(mapped_bfcl_tasks, "../data/bfcl_simple_mapped.json")

For evaluating with the custom tool set from BCFL, we modify the tool configuration from the nanobot:

In [ ]:
#| eval: false
from nanobot import Nanobot

bot = Nanobot.from_config()
# Tools of nanobot is saved in bot._loop.tools, which is a ToolManager object. We can get the tool definitions from there.
tool_registry = bot._loop.tools
tool_registry.get_definitions()[0]

{'type': 'function',
 'function': {'name': 'edit_file',
  'description': 'Edit a file by replacing old_text with new_text. Supports minor whitespace/line-ending differences. Set replace_all=true to replace every occurrence.',
  'parameters': {'type': 'object',
   'properties': {'path': {'type': 'string',
     'description': 'The file path to edit'},
    'old_text': {'type': 'string',
     'description': 'The text to find and replace'},
    'new_text': {'type': 'string', 'description': 'The text to replace with'},
    'replace_all': {'type': 'boolean',
     'description': 'Replace all occurrences (default false)'}},
   'required': ['path', 'old_text', 'new_text']}}}

Map tool function definitions (standard forms) into `nanobot` compatible `Tool` class:

In [ ]:
#| eval: false
from typing import Any, Callable, Type
from nanobot.agent.tools.base import Tool, tool_parameters
from nanobot.agent.tools.schema import *

test_function_schema = {
    "name": "test_function",
    "description": "Used to sanity-check function calling basic capability",
    "parameters": {
        "type": "object",
        "properties": {
            "sample_string": {
                "type": "string",
                "description": "A cool string to print out."
            },
            "sample_integer": {
                "type": "integer",
                "description": "An awesome number (a prime number)."
            },
            "sample_enum": {
                "type": "string",
                "enum": ["option_one", "option_two"],
                "description": "A parameter that only accepts specific predefined values."
            }
        },
        "required": ["sample_string", "sample_integer"]
    }
}

def _parse_schema_fragment(param_data: dict) -> Any:
    """Recursively parses a JSON schema dict into nanobot Schema instances."""
    if not param_data:
        return StringSchema("")

    p_type = param_data.get("type", "string")
    nullable = False
    
    # Handle ["type", "null"] pattern
    if isinstance(p_type, list):
        if "null" in p_type:
            nullable = True
        # Grab the primary type
        p_type = next((t for t in p_type if t != "null"), "string")

    p_desc = param_data.get("description", "")

    if p_type == "string":
        return StringSchema(
            description=p_desc,
            min_length=param_data.get("minLength"),
            max_length=param_data.get("maxLength"),
            enum=param_data.get("enum"),
            nullable=nullable
        )
        
    elif p_type == "integer":
        return IntegerSchema(
            description=p_desc,
            minimum=param_data.get("minimum"),
            maximum=param_data.get("maximum"),
            enum=param_data.get("enum"),
            nullable=nullable
        )
        
    elif p_type == "number":
        return NumberSchema(
            description=p_desc,
            minimum=param_data.get("minimum"),
            maximum=param_data.get("maximum"),
            enum=param_data.get("enum"),
            nullable=nullable
        )
        
    elif p_type == "boolean":
        return BooleanSchema(
            description=p_desc,
            default=param_data.get("default"),
            nullable=nullable
        )
        
    elif p_type == "array":
        items_data = param_data.get("items")
        parsed_items = _parse_schema_fragment(items_data) if items_data else None
        return ArraySchema(
            items=parsed_items,
            description=p_desc,
            min_items=param_data.get("minItems"),
            max_items=param_data.get("maxItems"),
            nullable=nullable
        )
        
    elif p_type == "object":
        raw_props = param_data.get("properties", {})
        parsed_props = {k: _parse_schema_fragment(v) for k, v in raw_props.items()}
        return ObjectSchema(
            properties=parsed_props,
            required=param_data.get("required"),
            description=p_desc,
            additional_properties=param_data.get("additionalProperties"),
            nullable=nullable
        )
        
    else:
        # Fallback for unknown types
        return StringSchema(description=p_desc, nullable=nullable)


def generate_tool_from_schema(
    schema: dict, 
    execute_fn: Callable, 
    base_class: Type[Tool] = Tool
) -> Type[Tool]:
    """
    Dynamically generates a nanobot Tool class from a standard JSON function schema.
    
    Args:
        schema: The JSON schema dict containing name, description, and parameters.
        execute_fn: An async function to be attached as the tool's 'execute' method. 
                    (Must accept 'self' as the first argument).
        base_class: The base class to inherit from (default is Tool).
    
    Returns:
        A dynamically generated, decorated Tool class ready for instantiation.
    """
    tool_name = schema.get("name", "dynamic_tool")
    tool_desc = schema.get("description", "")
    parameters = schema.get("parameters", {})
    properties = parameters.get("properties", {})
    required_params = parameters.get("required", [])

    # 1. Map top-level parameters using the recursive parser
    schema_kwargs = {}
    for param_name, param_data in properties.items():
        schema_kwargs[param_name] = _parse_schema_fragment(param_data)

    if required_params:
        schema_kwargs["required"] = required_params

    # 2. Build the root tool_parameters_schema
    t_schema = tool_parameters_schema(**schema_kwargs)

    # 3. Create property getters
    def _get_name(self) -> str:
        return tool_name

    def _get_description(self) -> str:
        return tool_desc

    # 4. Construct class dictionary
    class_dict = {
        "name": property(_get_name),
        "description": property(_get_description),
        "execute": execute_fn
    }
    
    if "read_only" in schema:
        class_dict["read_only"] = property(lambda self: schema["read_only"])

    # 5. Format class name dynamically
    class_name = "".join(word.capitalize() for word in tool_name.split("_")) + "Tool"

    # 6. Generate the class
    DynamicToolClass = type(class_name, (base_class,), class_dict)

    # 7. Apply the nanobot decorator
    return tool_parameters(t_schema)(DynamicToolClass)

async def test_result(self, sample_string: str, sample_integer: int, sample_enum: str = "option_one", **kwargs: Any) -> str:
    try:
        output_str = (
            f"Input string: {sample_string}\n"
            f"Input integer: {sample_integer}\n"
            f"Input option: {sample_enum}\n"
            "Test function executed correctly."
        )
        return output_str
    except Exception as e:
        return f"Error in running testing function: {e}"

# Generate the tool
TestTool = generate_tool_from_schema(test_function_schema, test_result)

Registering and test running with the new tool:

In [ ]:
#| eval: false
import json
tool_registry.register(TestTool())
definitions = tool_registry.get_definitions()
repasted_definition = [func for func in definitions if func['function']['name']=="test_function" ][0]
print(json.dumps(repasted_definition, indent=2))

{
  "type": "function",
  "function": {
    "name": "test_function",
    "description": "Used to sanity-check function calling basic capability",
    "parameters": {
      "type": "object",
      "properties": {
        "sample_string": {
          "type": "string",
          "description": "A cool string to print out."
        },
        "sample_integer": {
          "type": "integer",
          "description": "An awesome number (a prime number)."
        },
        "sample_enum": {
          "type": "string",
          "description": "A parameter that only accepts specific predefined values.",
          "enum": [
            "option_one",
            "option_two"
          ]
        }
      },
      "required": [
        "sample_string",
        "sample_integer"
      ]
    }
  }
}


In [ ]:
#| eval: false
result = await bot.run("Try to check the basic function calling capacity. Tell me the checking results.")
print(result.content)

2026-05-04 16:31:11.246 | INFO     | nanobot.agent.loop:_process_message:548 - Processing message from cli:user: Try to check the basic function calling capacity. Tell me the checking results.
2026-05-04 16:31:11.323 | DEBUG    | nanobot.agent.memory:maybe_consolidate_by_tokens:468 - Token consolidation idle sdk:default: 5493/131072 via tiktoken
2026-05-04 16:31:16.089 | INFO     | nanobot.agent.loop:before_execute_tools:96 - Tool call: test_function({"sample_string": "Hello, function calling!", "sample_integer": 42, "sample_enum": "option_one"})
2026-05-04 16:31:16.092 | DEBUG    | nanobot.agent.loop:after_iteration:101 - LLM usage: prompt=4096 completion=33 cached=0
2026-05-04 16:31:19.148 | DEBUG    | nanobot.agent.loop:after_iteration:101 - LLM usage: prompt=2803 completion=24 cached=0
2026-05-04 16:31:19.150 | INFO     | nanobot.agent.loop:_process_message:606 - Response to cli:user: Hello! I'm nanobot 🐈, your helpful AI assistant. How can I assist you today?
2026-05-04 16:31:19.2

Hello! I'm nanobot 🐈, your helpful AI assistant. How can I assist you today?


## tau-bench Task Selection

## 